## Starting Error Analysis
This section examines the learned feature weights of the Logistic Regression model to better understand what the classifier learned for each programming label.
Interpretations were supported by:
- The highest weighted TF-IDF features (tokens) learned for each class.
- Representative examples taken from the dataset containing those features.
> **Note:** During the exploratory analysis, all of the top-weighted features for each class were investigated. To keep this notebook concise, only representative feature investigations are shown, while the conclusions summarize evidence gathered from the full analysis.

In [1]:
import joblib

lr_model = joblib.load('../../models/logistic_regression_v1.joblib')
vectorizer = joblib.load('../../models/tfidf_vectorizer_v1.joblib')
print(f'Coefficient matrix shape: {lr_model.coef_.shape}')
print(f'Classes: {lr_model.classes_}')

Coefficient matrix shape: (5, 5793)
Classes: ['assignment' 'call' 'control_flow' 'expression' 'identifier']


In [2]:
feature_names = vectorizer.get_feature_names_out()

print(f'Classes: {lr_model.classes_}')
print(f'Coefficient matrix shape: {lr_model.coef_.shape}')
print(f'Number of features: {len(feature_names)}')

Classes: ['assignment' 'call' 'control_flow' 'expression' 'identifier']
Coefficient matrix shape: (5, 5793)
Number of features: 5793


In [3]:
print(lr_model.coef_.shape)
print(len(vectorizer.get_feature_names_out()))

(5, 5793)
5793


## Viewing the Weights for each Label
### Assignment Section
#### Assignment - Examples:

In [4]:
#rows in the model's matrix are 'assignment' 'call' 'control_flow' 'expression' 'identifier'
#since we are investigating assignment we are going to use the index 0
feature_names = vectorizer.get_feature_names_out()
assignment_row = lr_model.coef_[0]
top_indices = assignment_row.argsort()[-10:][::-1]
for i in top_indices:
    print(feature_names[i], assignment_row[i])

false 3.0205013214430614
none 1.588059164941726
flag 1.4932209190252854
decimal 1.4693535278723144
kabu 1.31569743850873
else 1.2452532654378823
approved 1.2118030758368012
round 1.1927882218464243
cache 1.1456162775748855
sorted 1.136313764612759


In [5]:
import pandas as pd

df = pd.read_csv('../../data/processed/cleaned_v1.csv')
filtered_df_assignment = df[(df['top_level_label'] == 'assignment') & (df['diff'].str.contains('approved'))]
filtered_assignment_5 = filtered_df_assignment.head(5)
display(filtered_assignment_5)

for index, row in filtered_assignment_5.iterrows():
    print(row['diff'])


,buggy_code,fixed_code,diff,top_level_label,full_label
10574,n = int(input())\nnums = [int(i) for i in inpu...,n = int(input())\nnums = [int(i) for i in inpu...,- approved = (num%3 == 0) or (num%5 == 0)\...,assignment,assignment.change


-     approved = (num%3 == 0) or (num%5 == 0)
+     approved = approved & ((num%3 == 0) or (num%5 == 0))


#### Assignment - Findings

- **Findings:**
- **Strongly Correlated:** No dominant strongly correlated tokens were observed.
- **Contextual Features:** `False`, `None`, `flag`, `else`
- **Context-Specific Programming Tokens:** `sorted`, `decimal`, `round`
- **Dataset Artifacts:** `kabu`, `approved`, `cache`
- Unlike the Control Flow and Call classes, Assignment did not exhibit many highly distinctive semantic features among its highest weighted tokens.
- Several of the highest weighted features were instead contextual indicators. For example, `flag` frequently appears in assignment-related code, making it a useful statistical indicator despite not defining the Assignment class itself.
- Some programming specific tokens, such as `sorted`, `decimal`, and `round`, received large weights because they commonly appeared in assignment-labeled examples. For example, `sorted` frequently occurs in statements such as `x = sorted(...)`, where the assignment and function call appear together. These tokens are meaningful programming concepts but are not unique indicators of the Assignment class.
- Several high-weighted features also reflected dataset-specific statistical correlations rather than the underlying programming concept. Tokens such as `kabu` and `cache` are highly specific to portions of the dataset, while `approved` appears to be a dataset artifact rather than a meaningful indicator of assignment. Although `approved` has semantic meaning within the individual program, there is no inherent relationship between the identifier name and the programming concept of assignment. Its high weight therefore likely reflects repeated lexical patterns in the training data rather than a generalizable feature of the Assignment class.

### Call Section
#### Call - Examples

In [6]:
#rows in the model's matrix are 'assignment' 'call' 'control_flow' 'expression' 'identifier'
#since we are investigating call we are going to use the index 1
call_row = lr_model.coef_[1]
top_indices = call_row.argsort()[-10:][::-1]
for i in top_indices:
    print(feature_names[i], call_row[i])

print 7.961567200417619
exit 4.132515132940391
quit 3.3731683893905258
sort 3.268155648621254
resolve 2.824162610965232
range 2.6023377075135268
split 2.317619365836531
006 2.218922004984276
reverse 2.135962712900584
use 2.0708382189820624


In [7]:
filtered_df_call = df[(df['top_level_label'] == 'call') & (df['diff'].str.contains('print'))]
filtered_call_5 = filtered_df_call.head(5)
display(filtered_call_5)

for index, row in filtered_call_5.iterrows():
    print(row['diff'])

,buggy_code,fixed_code,diff,top_level_label,full_label
0,"for i in range(1,10):\n for j in range(1,10...","for i in range(1,10):\n for j in range(1,10...","- print(i,""x"",j,""="",i*j)\n+ pr...",call,call.arguments.add
1,"[print('{}x{}={}').format(i, j, i * j) for j i...","[print('{}x{}={}'.format(i, j, i * j)) for i i...","- [print('{}x{}={}').format(i, j, i * j) for j...",call,call.arguments.change
3,"\nfor i in range(1,10):\n\tfor j in range(1,10...","for i in range(1,10):\n\tfor j in range(1,10):...","- \n- \t\tprint(i,""x"",j,""="",i*j,)\n+ \t\tprint...",call,call.arguments.keyword_argument.add
10,l = []\nfor i in range(10):\n l.append(int(...,l = []\nfor i in range(10):\n l.append(int(...,- l.append(int(input))\n+ l.append(int...,call,call.add
15,m=[input() for i in range(10)]\nm.sort()\nm.re...,#! /usr/bin/python3\n\nm=[int(input()) for i i...,+ #! /usr/bin/python3\n+ \n- m=[input() for i ...,call,call.arguments.change


-         print(i,"x",j,"=",i*j)
+         print(i,"x",j,"=",i*j,sep="")
- [print('{}x{}={}').format(i, j, i * j) for j in range(1, 10) for i in range(1, 10)]
+ [print('{}x{}={}'.format(i, j, i * j)) for i in range(1, 10) for j in range(1, 10)]
- 
- 		print(i,"x",j,"=",i*j,)
+ 		print(i,"x",j,"=",i*j,sep="")
-     l.append(int(input))
+     l.append(int(input()))
-     print(l[i])
+     print(l[9-i])
+ #! /usr/bin/python3
+ 
- m=[input() for i in range(10)]
+ m=[int(input()) for i in range(10)]
- print("{0}\n{1}\n{2}".format(m[0],m[1],m[2]))
+ print("{0}\n{1}\n{2}".format(m[0], m[1], m[2]))


#### Call - Findings

- **Findings:**
- **Strongly Correlated:** `print`, `exit`, `quit`, `sort`, `resolve`, `range`, `split`, `reverse`
- **Contextual Features:** No dominant contextual features were observed.
- **Dataset Artifacts:** `006`, `use`
- The Call class was highly interpretable. Many of its highest-weighted features corresponded to common Python function names, suggesting that the model learned meaningful lexical patterns associated with function call modifications.
- The model also assigned large weights to a small number of dataset-specific tokens. For example, `006` frequently appeared alongside modifications involving `abs()`, while `use` functioned as a variable rather than a meaningful indicator of function calls. These examples demonstrate that even highly interpretable classes can contain statistical correlations that are specific to the training data.

### Control Flow Section
#### Control Flow - Examples

In [8]:
#rows in the model's matrix are 'assignment' 'call' 'control_flow' 'expression' 'identifier'
#since we are investigating control flow we are going to use the index 2
control_row = lr_model.coef_[2]
top_indices = control_row.argsort()[-10:][::-1]
for i in top_indices:
    print(feature_names[i], control_row[i])

if 7.0016018894330765
break 6.847030286410828
elif 5.329618127652205
or 4.4122867784809285
while 4.002098118143551
and 3.510672766352269
continue 3.3832527516880244
return 3.2617027332453272
not 2.5902295397346777
except 2.2773409803486966


In [9]:
filtered_df_control = df[(df['top_level_label'] == 'control_flow') & (df['diff'].str.contains('if'))]
filtered_control_5 = filtered_df_control.head(5)
display(filtered_control_5)

for index, row in filtered_control_5.iterrows():
    print(row['diff'])

,buggy_code,fixed_code,diff,top_level_label,full_label
38,N = int(input())\n\nfor i in range(N):\n a ...,N = int(input())\n\nfor i in range(N):\n a ...,- if a[0]^2 + a[1]^2 == a[2]^2:\n+ if ...,control_flow,control_flow.branch.if.condition.change
48,n=int(input())\nfor i in range(n):\n\tA=[int(j...,n=int(input())\nfor i in range(n):\n\tA=[int(j...,+ \tA.sort()\n- \tif(A[0]**2+A[1]**2==A[2]**2)...,control_flow,control_flow.branch.if.condition.change
50,n = int(input())\nL = [input() for i in range(...,n = int(input())\nL = [input() for i in range(...,- L = [input() for i in range(n)] #L?????????...,control_flow,control_flow.branch.if.condition.change
150,"import sys\n\ndrops = [None,\n [(0,0),...","import sys\n\ndrops = [None,\n [(0,0),...",- if (0 <= nx <= 9 or 0 <= ny <= 9):\n...,control_flow,control_flow.branch.if.condition.change
172,import sys\nfrom itertools import combinations...,import sys\nfrom itertools import combinations...,- if k and v:\n+ if k or v:,control_flow,control_flow.branch.if.condition.change


-     if a[0]^2 + a[1]^2 == a[2]^2:
+     if a[0]**2 + a[1]**2 == a[2]**2:
+ 	A.sort()
- 	if(A[0]**2+A[1]**2==A[2]**2):
+ 	if A[0]**2+A[1]**2==A[2]**2:
- L = [input() for i in range(n)]  #L????????????????´???¨???????????????
+ L = [input() for i in range(n)]
- 	A = line.split()  #A???line??????3????????°???????????????(?????????)
- 	B = sorted([int(A[i]) for i in range(3)])  #B???A?????°?????????????????????????????????
+ 	A = line.split()
+ 	B = sorted([int(A[i]) for i in range(3)])
- 	if B[2]^2 == B[0]^2 + B[1]^2:
+ 	if B[2]**2 == B[0]**2 + B[1]**2:
-         if (0 <= nx <= 9 or 0 <= ny <= 9):
+         if (0 <= nx <= 9 and 0 <= ny <= 9):
-     if k and v:
+     if k or v:


#### Control Flow - Findings

- **Findings:**
- **Strongly Correlated:** `if`, `elif`, `break`, `or`, `while`, `and`, `continue`, `return`, `not`, `except`
- **Contextual Features:** No dominant contextual features were observed.
- **Dataset Artifacts:** No dominant dataset artifacts were observed.
- The highest weighted features were dominated by common Python control flow keywords, making Control Flow the most interpretable class. Unlike Assignment and Expression, there was little evidence of contextual features or dataset artifacts among the strongest coefficients.
- The strong correspondence between the learned features and Python control flow keywords suggests that the model learned meaningful lexical patterns associated with control flow modifications rather than relying on dataset-specific statistical correlations.

### Expression Section
#### Expression - Examples

In [10]:
#rows in the model's matrix are 'assignment' 'call' 'control_flow' 'expression' 'identifier'
#since we are investigating expression we are going to use the index 3
expression_row = lr_model.coef_[3]
top_indices = expression_row.argsort()[-10:][::-1]
for i in top_indices:
    print(feature_names[i], expression_row[i])

print 6.258444542726909
range 2.9959366553280686
safe 1.7724317346864753
180 1.7139073022385372
union 1.672559309429109
taxi 1.4763991229654134
if 1.4715791842465373
mod 1.4128835829410675
append 1.3794671666593201
26 1.3706714636777317


In [11]:
filtered_df_expression = df[(df['top_level_label'] == 'expression') & (df['diff'].str.contains('safe'))]
filtered_expression_5 = filtered_df_expression.head(5)
display(filtered_expression_5)

for index, row in filtered_expression_5.iterrows():
    print(row['diff'])

,buggy_code,fixed_code,diff,top_level_label,full_label
2619,def q4():\n n = int(input())\n line = in...,def q4():\n n = int(input())\n line = in...,- safe = line[0:r-1].count('R')\n+ ...,expression,expression.operation.binary.remove
6891,"s, w = map(int, input().split())\nans = ""safe""...","s, w = map(int, input().split())\nans = ""safe""...","- ans = ""safe"" if s > w * 2 else 'unsafe'\n+ a...",expression,expression.operation.binary.remove
6903,"S,W=map(int,input().split())\nprint(""unsafe"" i...","S, W = map(int, input().split())\nprint(""unsaf...","- S,W=map(int,input().split())\n+ S, W = map(i...",expression,expression.operation.binary.remove
6904,"S,W=map(int,input().split())\nprint(""unsafe"" i...","S, W = map(int, input().split())\nprint(""unsaf...","- S,W=map(int,input().split())\n+ S, W = map(i...",expression,expression.operation.binary.remove
6905,"S,W=map(int,input().split())\nprint(""unsafe"" i...","S,W=map(int,input().split())\nprint(""unsafe"" i...","- print(""unsafe"" if W>=S//2 else ""safe"")\n+ pr...",expression,expression.operation.binary.remove


-         safe = line[0:r-1].count('R')
+         safe = line[0:r].count('R')
+ 
+ 
- ans = "safe" if s > w * 2 else 'unsafe'
+ ans = "safe" if s > w else 'unsafe'
- S,W=map(int,input().split())
+ S, W = map(int, input().split())
- print("unsafe" if W>=S//2 else "safe")
+ print("unsafe" if W >= S else "safe")
- S,W=map(int,input().split())
+ S, W = map(int, input().split())
- print("unsafe" if W>=S/2 else "safe")
+ print("unsafe" if W >= S else "safe")
- print("unsafe" if W>=S//2 else "safe")
+ print("unsafe" if W>=S else "safe")


#### Expression - Findings 
- Findings:
- Strongly Correlated: No dominant strongly correlated tokens were found.
- Contextual Features: `print`, `range`, `if`, `append`
- Dataset Artifacts: `safe`, `180`, `union`, `taxi`, `mod`, `26`
- The model primarily relied on contextual programming tokens such as `print`, `range`, `if`, and `append` rather than highly distinctive expression-specific features. These tokens frequently surround or contain expression modifications, making them useful statistical indicators despite not defining the Expression class.
- Inspection of representative examples showed that many expression modifications occurred within larger programming concepts, such as function calls. For example, the surrounding `print()` call often remained unchanged while only the expression passed as an argument was modified. Because TF-IDF represents the entire diff as a bag of tokens, it cannot distinguish between tokens belonging to the unchanged function call and those belonging to the modified expression.
- This likely contributed to the stronger performance of the Call class relative to the Expression class, as call-related tokens remained highly visible even when the underlying AST modification belonged to an expression.
- This illustrates a fundamental limitation of the TF-IDF representation. While it captures which tokens appear in a diff, it does not preserve the structural context of those tokens, making it difficult to distinguish expression modifications from the surrounding programming constructs in which they occur.
- Several high-weighted features appeared to be dataset-specific artifacts rather than meaningful indicators of the Expression class, suggesting that the model relied on lexical correlations when distinctive expression features were unavailable.

### Identifier Section
#### Identifier - Examples

In [12]:
#rows in the model's matrix are 'assignment' 'call' 'control_flow' 'expression' 'identifier'
#since we are investigating identifier we are going to use the index 4
identifier_row = lr_model.coef_[4]
top_indices = identifier_row.argsort()[-10:][::-1]
for i in top_indices:
    print(feature_names[i], identifier_row[i])

appendleft 4.068687062560456
def 3.478799978405608
import 3.350273689293935
for 2.9004898161295976
from 2.7728031018548815
reverse 2.6750746387068887
appned 2.628422948904686
add 2.5895885697833902
itertools 2.219008135391293
rslt 1.9246239712951005


In [13]:
filtered_df_identifier = df[(df['top_level_label'] == 'identifier') & (df['diff'].str.contains('appendleft'))]
filtered_identifier_5 = filtered_df_identifier.head(5)
display(filtered_identifier_5)

for index, row in filtered_identifier_5.iterrows():
    print(str(index) + ' row')
    print(row['diff'])

,buggy_code,fixed_code,diff,top_level_label,full_label
603,from collections import deque\nl=deque()\nfor ...,from collections import deque\nl=deque()\nfor ...,"- elif c==""s"": l.extendleft(a[1])\n+ e...",identifier,identifier.change
618,from collections import deque\nquery = int(inp...,from collections import deque\nquery = int(inp...,- command = input()\n+ command = input...,identifier,identifier.change
625,from collections import deque\nn = int(input()...,from collections import deque\nn = int(input()...,"+ a.popleft()\n+ elif c == ""delete...",identifier,identifier.change
626,from collections import deque\nn = int(input()...,from collections import deque\nn = int(input()...,- a.extendleft(v)\n+ a...,identifier,identifier.change
5465,from collections import deque\n\n\ndef solve()...,from collections import deque\n\n\ndef solve()...,- \n+ \n- que.appendleft(nv)\n...,identifier,identifier.change


603 row
-     elif c=="s": l.extendleft(a[1])
+     elif c=="s": l.appendleft(a[1])
618 row
-     command = input()
+     command = input().strip()
-             l.append(int(value))
+             l.appendleft(int(value))
625 row
+         a.popleft()
+     elif c == "deleteLast":
-     elif c == "deleteLast":
-         a.popleft()
-             a.extendleft(v)
+             a.appendleft(v)
626 row
-             a.extendleft(v)
+             a.appendleft(v)
5465 row
-     
+ 
-             que.appendleft(nv)
+             que.append(nv)


#### Identifier - Findings

- **Findings:**
- **Strongly Correlated:** No dominant strongly correlated tokens were observed.
- **Contextual Features:** `def`, `import`, `for`, `from`
- **Context-Specific Programming Tokens:** `appendleft`, `add`, `itertools`
- **Dataset Artifacts:** `appned`
- Manual inspection revealed that many Identifier examples were initially difficult to interpret from a human perspective. However, the RunBugRun dataset assigns labels based on syntactic AST changes rather than human semantic interpretation.
- **Representative Example:**
  ```python
  - elif c == "s": l.extendleft(a[1])
  + elif c == "s": l.appendleft(a[1])
  ```
- Although many developers would describe this as a function call modification, the changed AST node is the function identifier (`extendleft` → `appendleft`). Under RunBugRun's labeling methodology, this is therefore classified as an Identifier modification.
- Overall, the Identifier class learned useful statistical correlations involving contextual programming keywords such as `def`, `import`, `for`, and `from`. For example, examples containing `import` and `from` often had identifier changes caused by misspelled module or imported names, while examples containing `for` frequently involved changes to identifiers used inside loop constructs.
- In addition, the model was able to learn from context-specific programming tokens like `appendleft` and `itertools`. Rather than understanding the semantic difference between these changes, it learned the statistical correlation between specific identifier modifications that repeatedly appeared in the dataset.
- This class demonstrates one of the limitations of TF-IDF. Correctly predicting identifier changes often requires understanding the surrounding code and the role an identifier plays, something that is lost in a bag-of-words representation. This works well for classes like Control Flow where the set of keywords is very predictable, but Identifier changes require much more context.